# Generative AI and Large Language Model Foundations

**Production unit:** PU-B04-C01  
**Canonical companion:** M4_N01_generative_ai_llm_foundations.ipynb  
**Version:** 0.1 candidate

This laboratory develops a transparent miniature language-generation pipeline. It does not reproduce a frontier model; it exposes the mechanisms needed to reason about tokens, conditional probability, decoding, grounding, evaluation, and accountable institutional use.

## Learning objectives

By completing the notebook, you will be able to:

1. distinguish generative and discriminative modelling;
2. construct and inspect a small tokenizer;
3. estimate next-token probabilities from a corpus;
4. compare greedy, temperature, top-k, and top-p decoding;
5. compute a simplified self-attention operation;
6. distinguish fluency, correctness, groundedness, safety, and decision fitness;
7. document a reproducible SRAI evidence chain.

## Laboratory map

| Part | Question |
|---|---|
| A | How does text become model input? |
| B | How is a next-token distribution estimated? |
| C | How do decoding choices change generated sequences? |
| D | How does self-attention combine contextual information? |
| E | Why must generated claims be grounded and evaluated? |
| F | What evidence is required before institutional use? |

In [ ]:
# Environment and reproducibility
from collections import Counter, defaultdict
import math
import platform
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)
print({"python": sys.version.split()[0], "platform": platform.platform(), "seed": SEED})

## Part A — Tokens and representations

A tokenizer is part of the model contract. Token identifiers have meaning only relative to the exact tokenizer and vocabulary used. The simple tokenizer below is intentionally visible and auditable.

In [ ]:
def tokenize(text):
    """Lowercase and split words and punctuation for demonstration."""
    return re.findall(r"\w+(?:'\w+)?|[^\w\s]", text.lower(), flags=re.UNICODE)

examples = [
    "Generative AI predicts and creates new content.",
    "Large language models operate over tokens.",
    "Context influences the next-token distribution.",
]

for text in examples:
    print(text)
    print(tokenize(text))

In [ ]:
SPECIAL = ["<pad>", "<unk>", "<bos>", "<eos>"]
counts = Counter(token for text in examples for token in tokenize(text))
vocabulary = SPECIAL + sorted(counts)
token_to_id = {token: i for i, token in enumerate(vocabulary)}
id_to_token = {i: token for token, i in token_to_id.items()}

pd.DataFrame({
    "token": vocabulary,
    "token_id": range(len(vocabulary)),
    "corpus_count": [counts.get(t, 0) for t in vocabulary],
}).head(20)

In [ ]:
def encode(text):
    ids = [token_to_id["<bos>"]]
    ids += [token_to_id.get(t, token_to_id["<unk>"]) for t in tokenize(text)]
    ids += [token_to_id["<eos>"]]
    return ids

def decode(ids):
    ignored = {token_to_id["<pad>"], token_to_id["<bos>"], token_to_id["<eos>"]}
    return " ".join(id_to_token.get(i, "<unk>") for i in ids if i not in ignored)

encoded = encode(examples[0])
print("Encoded:", encoded)
print("Decoded:", decode(encoded))
print("Unknown-token example:", encode("Auditable systems support institutions."))

### Interpretation

The same visible text can produce different token counts under different tokenizers. Token count affects context use, latency, and cost. Unknown-token behaviour also reveals whether a vocabulary can represent the intended language and domain.

In [ ]:
token_summary = pd.DataFrame({
    "text": examples,
    "characters": [len(x) for x in examples],
    "tokens": [len(tokenize(x)) for x in examples],
    "unique_tokens": [len(set(tokenize(x))) for x in examples],
})
token_summary

In [ ]:
ax = token_summary.plot.bar(x="text", y="tokens", legend=False, color="#2563A6", figsize=(8, 3))
ax.set_ylabel("Token count")
ax.set_xlabel("")
ax.set_title("Token counts under the demonstration tokenizer")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## Part B — Autoregressive probability

For a token sequence $x_1,\ldots,x_T$, an autoregressive model factorizes sequence probability as

$$p(x_1,\ldots,x_T)=\prod_{t=1}^{T}p(x_t\mid x_1,\ldots,x_{t-1}).$$

The miniature bigram model below approximates the next token using only the immediately preceding token. A real LLM conditions on a much richer context.

In [ ]:
training_sentences = [
    "evidence supports a decision",
    "authorized evidence supports an accountable decision",
    "verified evidence supports an institutional decision",
    "a model generates a probable continuation",
    "a model output requires verification",
    "human review supports accountable use",
]

bigram_counts = defaultdict(Counter)
for sentence in training_sentences:
    sequence = ["<bos>"] + tokenize(sentence) + ["<eos>"]
    for current, nxt in zip(sequence, sequence[1:]):
        bigram_counts[current][nxt] += 1

def next_token_distribution(current, alpha=0.1):
    candidates = sorted({t for counter in bigram_counts.values() for t in counter})
    raw = np.array([bigram_counts[current][t] + alpha for t in candidates], dtype=float)
    return candidates, raw / raw.sum()

tokens, probabilities = next_token_distribution("evidence")
pd.DataFrame({"next_token": tokens, "probability": probabilities}).sort_values("probability", ascending=False).head(8)

In [ ]:
top = pd.DataFrame({"token": tokens, "probability": probabilities}).nlargest(8, "probability")
plt.figure(figsize=(7, 3))
plt.bar(top["token"], top["probability"], color="#2F6F4E")
plt.ylabel("Probability")
plt.title("Estimated next-token distribution after 'evidence'")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
def sequence_log_probability(sentence, alpha=0.1):
    sequence = ["<bos>"] + tokenize(sentence) + ["<eos>"]
    total = 0.0
    for current, nxt in zip(sequence, sequence[1:]):
        candidates, probs = next_token_distribution(current, alpha)
        lookup = dict(zip(candidates, probs))
        total += math.log(lookup.get(nxt, 1e-12))
    return total

candidates = [
    "evidence supports a decision",
    "a model output requires verification",
    "evidence guarantees truth",
]
pd.DataFrame({"sequence": candidates, "log_probability": [sequence_log_probability(x) for x in candidates]})

### Reasoning checkpoint

A higher probability means a sequence is more compatible with the demonstration corpus. It does **not** prove that the statement is factually correct, safe, or authorized for a decision.

## Part C — Decoding is a policy choice

The model returns probabilities; a decoding rule chooses tokens. Greedy decoding is deterministic. Sampling can increase diversity but also variation and risk. Temperature rescales logits, while top-k and top-p restrict the candidate set.

In [ ]:
def apply_temperature(probabilities, temperature=1.0):
    if temperature <= 0:
        raise ValueError("temperature must be positive")
    logits = np.log(np.clip(probabilities, 1e-12, 1.0)) / temperature
    weights = np.exp(logits - logits.max())
    return weights / weights.sum()

base = np.array([0.55, 0.25, 0.12, 0.08])
labels = np.array(["verified", "probable", "fluent", "unsupported"])
temperature_table = pd.DataFrame({
    "token": labels,
    "T=0.5": apply_temperature(base, 0.5),
    "T=1.0": apply_temperature(base, 1.0),
    "T=1.5": apply_temperature(base, 1.5),
})
temperature_table

In [ ]:
temperature_table.set_index("token").plot.bar(figsize=(8, 3), color=["#1F4E79", "#5B9BD5", "#A5A5A5"])
plt.ylabel("Probability")
plt.title("Temperature redistributes probability mass")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
def restrict_top_k(tokens, probabilities, k):
    order = np.argsort(probabilities)[::-1]
    keep = order[:k]
    p = probabilities[keep]
    return tokens[keep], p / p.sum()

def restrict_top_p(tokens, probabilities, p_threshold):
    order = np.argsort(probabilities)[::-1]
    sorted_p = probabilities[order]
    count = np.searchsorted(np.cumsum(sorted_p), p_threshold, side="left") + 1
    keep = order[:count]
    p = probabilities[keep]
    return tokens[keep], p / p.sum()

print("top-k:", restrict_top_k(labels, base, 2))
print("top-p:", restrict_top_p(labels, base, 0.80))

In [ ]:
def select_next(current, method="greedy", temperature=1.0, k=3, p_threshold=0.9, generator=None):
    generator = generator or np.random.default_rng(SEED)
    tokens, probs = next_token_distribution(current)
    tokens = np.array(tokens)
    probs = apply_temperature(probs, temperature)
    if method == "greedy":
        return tokens[np.argmax(probs)]
    if method == "top_k":
        tokens, probs = restrict_top_k(tokens, probs, k)
    elif method == "top_p":
        tokens, probs = restrict_top_p(tokens, probs, p_threshold)
    return generator.choice(tokens, p=probs)

for method in ["greedy", "top_k", "top_p"]:
    local_rng = np.random.default_rng(SEED)
    print(method, [select_next("evidence", method=method, generator=local_rng) for _ in range(8)])

### Governance implication

Decoding settings are part of the evidence record. If a consequential output cannot be reproduced exactly, the organization should at least retain the model version, prompt, context, settings, time, and verification outcome.

## Part D — A transparent self-attention calculation

The Transformer introduced an architecture based on attention mechanisms. In scaled dot-product attention,

$$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V.$$

This example uses tiny matrices so every operation can be inspected.

In [ ]:
def softmax(x, axis=-1):
    shifted = x - np.max(x, axis=axis, keepdims=True)
    ex = np.exp(shifted)
    return ex / ex.sum(axis=axis, keepdims=True)

X = np.array([
    [1.0, 0.0, 1.0],   # evidence
    [0.0, 1.0, 1.0],   # supports
    [1.0, 1.0, 0.0],   # decision
])
Q = K = V = X
scores = Q @ K.T / np.sqrt(K.shape[1])
weights = softmax(scores, axis=1)
contextualized = weights @ V

pd.DataFrame(weights, index=["evidence", "supports", "decision"], columns=["evidence", "supports", "decision"])


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
image = ax.imshow(weights, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(3), ["evidence", "supports", "decision"])
ax.set_yticks(range(3), ["evidence", "supports", "decision"])
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{weights[i,j]:.2f}", ha="center", va="center")
ax.set_title("Demonstration self-attention weights")
fig.colorbar(image, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

### Interpretation boundary

Attention weights show how this calculation combines value vectors. They are not automatically a causal explanation, a factual citation, or proof of valid reasoning.

## Part E — Grounding and evaluation

Generated content should be evaluated along distinct dimensions. A fluent answer can be unsupported; a supported answer can still be incomplete; a correct answer can be unsuitable for an institutional decision.

In [ ]:
authorized_evidence = {
    "revenue": "Quarterly revenue was USD 12.4 million.",
    "period": "The reporting period ended 30 June 2026.",
    "status": "The figures remain provisional pending audit.",
}

candidate_outputs = [
    "Quarterly revenue was USD 12.4 million and remains provisional pending audit.",
    "Quarterly revenue was USD 14.2 million and has been audited.",
    "The report presents quarterly revenue figures.",
]

required_claims = {
    "12.4 million": authorized_evidence["revenue"],
    "30 june 2026": authorized_evidence["period"],
    "provisional": authorized_evidence["status"],
    "pending audit": authorized_evidence["status"],
}

def evidence_support(output):
    lower = output.lower()
    expressed = [claim for claim in required_claims if claim in lower]
    unsupported_flags = [flag for flag in ["14.2 million", "has been audited"] if flag in lower]
    return {
        "expressed_supported_claims": len(expressed),
        "unsupported_claim_flags": len(unsupported_flags),
        "grounded": len(unsupported_flags) == 0 and len(expressed) > 0,
    }

evaluation = pd.DataFrame([{"output": x, **evidence_support(x)} for x in candidate_outputs])
evaluation

In [ ]:
dimensions = pd.DataFrame({
    "dimension": ["fluency", "groundedness", "correctness", "safety", "decision_fitness"],
    "question": [
        "Is the output coherent and relevant?",
        "Are material claims supported by authorized evidence?",
        "Do claims agree with verified reality?",
        "Does the output avoid prohibited or harmful behaviour?",
        "Is the output suitable for this workflow and accountable use?",
    ],
    "evidence": [
        "human or rubric review",
        "claim-to-source trace",
        "verified reference values",
        "scenario and policy tests",
        "use-case validation and human authorization",
    ],
})
dimensions

### Public-enterprise case

An analyst asks a model to summarize a quarterly report. The generated answer states the wrong financial figure with confident language. The appropriate control is not simply “use a better prompt.” The workflow needs authorized source selection, claim-level verification, provisional-status preservation, human review, and a retained evidence record.

### Official-statistics case

A generated agricultural-statistics narrative must preserve the indicator definition, unit, geography, reference period, population, revision status, and uncertainty. A numerically correct value becomes misleading if separated from any of these qualifiers.

In [ ]:
evidence_chain = pd.DataFrame([
    {"stage": 1, "record": "request", "example": "Summarize provisional quarterly revenue"},
    {"stage": 2, "record": "authorized context", "example": "Approved report and metadata"},
    {"stage": 3, "record": "model and settings", "example": "Version, prompt, temperature, timestamp"},
    {"stage": 4, "record": "generated output", "example": "Unedited model response"},
    {"stage": 5, "record": "verification", "example": "Claim-source comparison and exceptions"},
    {"stage": 6, "record": "human decision", "example": "Approve, revise, reject, or escalate"},
])
evidence_chain

## Part F — Decision fitness

Use the following checklist before generated content influences a consequential workflow. Technical capability does not establish institutional authorization.

In [ ]:
decision_checklist = pd.DataFrame({
    "control": [
        "Intended use is documented",
        "Excluded uses are documented",
        "Sources are authorized and current",
        "Material claims are traceable",
        "Model and decoding settings are recorded",
        "Privacy and security controls are applied",
        "Failure and escalation paths exist",
        "An accountable human authorizes use",
    ],
    "status": ["Required"] * 8,
})
decision_checklist

## Exercises

1. Add a multilingual sentence and inspect token fragmentation.
2. Add two conflicting training sentences and explain the changed next-token distribution.
3. Compare sequences sampled at temperatures 0.5, 1.0, and 1.5.
4. Modify the attention inputs and explain which weights change.
5. Create an unsupported candidate output and test the evidence-support function.
6. Extend the evidence chain with retention period, reviewer identity, and incident reference.
7. Explain why a high-probability sequence is not necessarily true.
8. Draft an excluded-use statement for an official-statistics publication workflow.

In [ ]:
# Learner workspace: Exercise 1
multilingual_text = "Les statistiques agricoles nécessitent des métadonnées vérifiables."
print(tokenize(multilingual_text))
print("Token count:", len(tokenize(multilingual_text)))

In [ ]:
# Learner workspace: Exercise 3
for temperature in [0.5, 1.0, 1.5]:
    adjusted = apply_temperature(base, temperature)
    print(temperature, dict(zip(labels, np.round(adjusted, 3))))

## Knowledge check

1. What distinguishes a generative model from a discriminative model?
2. What does an autoregressive language model estimate at each generation step?
3. Why must training, adaptation, inference, retrieval, and tool use be distinguished?
4. Which dimensions should be evaluated before consequential use?
5. Which provenance and governance information should accompany a deployed LLM application?

## Instructor solution outline

1. A generative model learns a distribution from which new outputs can be produced; a discriminative model estimates labels or outcomes conditional on observed features.
2. It estimates a conditional probability distribution over the next token given the retained context.
3. They create different evidence, control, security, and governance requirements.
4. At minimum: fluency, groundedness, correctness, robustness, safety, subgroup behaviour, cost, latency, and decision fitness.
5. Record model and data provenance, intended and excluded uses, evaluation evidence, prompt and context policy, decoding settings, safeguards, limitations, residual risks, and accountable approval.

## Key conclusions

- Language generation is repeated conditional prediction, not retrieval of guaranteed truth.
- Tokenization, context, model version, and decoding settings shape outputs.
- Attention enables contextual combination but is not itself a factual citation or complete explanation.
- Grounding and verification supply authority that fluency cannot.
- Consequential use requires an evidence chain and accountable human authorization.

**SRAI Principle No. 96:** A language model generates statistically plausible continuations; factual authority must come from evidence, validation, and accountable human use.

## References

- Vaswani, A. et al. (2017). *Attention Is All You Need*. arXiv:1706.03762.
- Brown, T. B. et al. (2020). *Language Models are Few-Shot Learners*. arXiv:2005.14165.
- Lewis, P. et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. arXiv:2005.11401.
- National Institute of Standards and Technology (2024). *Artificial Intelligence Risk Management Framework: Generative Artificial Intelligence Profile*. NIST AI 600-1.

These sources establish the technical and governance foundations. The demonstrations in this notebook are intentionally small and educational.